# Imports

In [1]:
import pickle
import optuna

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_90_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_90_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004927,0.001350,0.993723,0.003345,0.002030,0.994625,0.008438,1.091890e-03,0.990470
1,0.996682,0.000414,0.002905,0.996100,0.000521,0.003379,0.993015,2.545769e-04,0.006731
2,0.003205,0.000699,0.996095,0.002819,0.000623,0.996558,0.003345,8.700245e-04,0.995785
3,0.004065,0.003263,0.992672,0.003906,0.002117,0.993978,0.004627,1.166784e-03,0.994206
4,0.996028,0.000021,0.003951,0.995773,0.000024,0.004204,0.990619,7.828402e-07,0.009381


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.010078,0.003001,0.986921,0.009916,0.003474,0.986609,0.009972,0.003525,0.986503
1,0.457102,0.002879,0.540019,0.505986,0.002978,0.491037,0.516644,0.001511,0.481845
2,0.998144,0.001284,0.000572,0.996786,0.002227,0.000987,0.998283,0.001372,0.000345
3,0.988863,0.000017,0.011120,0.989342,0.000024,0.010634,0.991011,0.000050,0.008939
4,0.006397,0.002261,0.991343,0.006481,0.001816,0.991703,0.004760,0.001775,0.993465


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [14]:
def objective(trial, X, y):
    
    C = trial.suggest_float("C", 1e-5, 100.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    tol = trial.suggest_float("tol", 1e-4, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        model = LogisticRegression(
            solver="lbfgs",
            l1_ratio=0.0,
            C=C,
            class_weight=class_weight,
            fit_intercept=fit_intercept,
            tol=tol,
            max_iter=max_iter,
            random_state=42,
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)
        
        weighted_probas = proba * weights
        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=60, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-08 16:41:12,511] A new study created in memory with name: no-name-e4442cd1-cb66-4694-8f96-b94d9a0ca176
Best trial: 5. Best value: 0.587084:   2%|██▎                                                                                                                                       | 1/60 [00:11<10:58, 11.16s/it]

[I 2026-07-08 16:41:23,649] Trial 5 finished with value: 0.5870837854071609 and parameters: {'C': 1.8799000875026933e-05, 'class_weight': None, 'fit_intercept': False, 'tol': 0.0034687427676954835, 'max_iter': 3188, 'weight_class_0': 0.7920368357377003, 'weight_class_1': 0.8455808762707437, 'weight_class_2': 0.2419388857433938}. Best is trial 5 with value: 0.5870837854071609.


Best trial: 5. Best value: 0.587084:   3%|████▌                                                                                                                                     | 2/60 [00:12<05:26,  5.62s/it]

[I 2026-07-08 16:41:25,415] Trial 0 finished with value: 0.580607738450374 and parameters: {'C': 6.525088353636968e-05, 'class_weight': None, 'fit_intercept': False, 'tol': 0.0026270826372161407, 'max_iter': 3421, 'weight_class_0': 7.804049660204241, 'weight_class_1': 0.3066338906504279, 'weight_class_2': 1.4796396787848003}. Best is trial 5 with value: 0.5870837854071609.


Best trial: 9. Best value: 0.601187:   5%|██████▉                                                                                                                                   | 3/60 [00:14<03:43,  3.91s/it]

...

[I 2026-07-08 16:42:55,416] Trial 59 finished with value: 0.9485251581227863 and parameters: {'C': 0.002137650570521202, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0005384889473122052, 'max_iter': 3253, 'weight_class_0': 1.0393965014516309, 'weight_class_1': 3.2971362724093587, 'weight_class_2': 4.40528821415475}. Best is trial 31 with value: 0.9497128324966972.


Best trial: 31. Best value: 0.949713:  98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 59/60 [01:43<00:00,  1.15it/s]

[I 2026-07-08 16:42:55,702] Trial 57 finished with value: 0.9486380996700869 and parameters: {'C': 0.001958063839373326, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0002556299431989669, 'max_iter': 3159, 'weight_class_0': 1.0295950551508368, 'weight_class_1': 3.3827403652227197, 'weight_class_2': 4.57378436184786}. Best is trial 31 with value: 0.9497128324966972.


Best trial: 31. Best value: 0.949713: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [01:43<00:00,  1.72s/it]

[I 2026-07-08 16:42:56,004] Trial 58 finished with value: 0.9496377640824301 and parameters: {'C': 0.0024640642231709823, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00024811751530652104, 'max_iter': 3205, 'weight_class_0': 0.6116402827401176, 'weight_class_1': 3.410585055192493, 'weight_class_2': 4.332224849661684}. Best is trial 31 with value: 0.9497128324966972.

Best trial score:
0.9497128324966972

Best params:
{'C': 0.01798806694620488, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00010184796505166689, 'max_iter': 3649, 'weight_class_0': 0.29255523927819943, 'weight_class_1': 1.8148882329644411, 'weight_class_2': 2.119330647860387}


In [15]:
lg_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(
    solver="lbfgs",
    l1_ratio=0.0,
    random_state=42,
    **lg_params
).fit(X_train, y_train.health_condition)

test_proba = lg.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [16]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [17]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.4.2_xgboost_lightgbm_catboost_stacking_submission.csv', index=False)

In [18]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [19]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'xgb_0', 'xgb_1', 'xgb_2', 'cat_0',
       'cat_1', 'cat_2'],
      dtype='str')

In [20]:
len(study.trials)

60